In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-5 : GET RAW INVALID RECORDS FROM QUARANTINE
# =========================================================

invalid_df = spark.read.table("retails.silver.order_items_quarantine") \
                    .filter((col("_rescued_data").isNotNull()) & (col("quarantine_status") == 'NEW'))

In [0]:
from pyspark.sql.functions import col, get_json_object as get_json_from

# =========================================================
# STEP-6 : TRY TO RECOVER RESCUED COLUMNS
# =========================================================
recovered_df = invalid_df \
                .withColumn("order_item_id_fixed", get_json_from(col("_rescued_data"), "$.order_item_id")) \
                .withColumn("order_item_order_id_fixed", get_json_from(col("_rescued_data"), "$.order_item_order_id")) \
                .withColumn("order_item_product_id_fixed", get_json_from(col("_rescued_data"), "$.order_item_product_id")) \
                .withColumn("order_item_quantity_fixed", get_json_from(col("_rescued_data"), "$.order_item_quantity")) \
                .withColumn("order_item_subtotal_fixed", get_json_from(col("_rescued_data"), "$.order_item_subtotal")) \
                .withColumn("order_item_product_price_fixed", get_json_from(col("_rescued_data"), "$.order_item_product_price"))


In [0]:
from pyspark.sql.functions import coalesce, trim

# =========================================================
# STEP-7 : MERGE RECOVERED VALUES
# =========================================================

recovered_df = recovered_df \
    .withColumn(
        "order_item_id",
        coalesce(col("order_item_id"), col("order_item_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "order_item_order_id",
        coalesce(col("order_item_order_id"), trim(col("order_item_order_id_fixed")))
    ) \
    .withColumn(
        "order_item_product_id",
        coalesce(col("order_item_product_id"), col("order_item_product_id_fixed").cast("double"))
    ) \
    .withColumn(
        "order_item_quantity",
        coalesce(col("order_item_quantity"), col("order_item_quantity_fixed").cast("bigint"))
    ) \
    .withColumn(
        "order_item_subtotal",
        coalesce(col("order_item_subtotal"), col("order_item_subtotal_fixed").cast("bigint"))
    ) \
    .withColumn(
        "order_item_product_price",
        coalesce(col("order_item_product_price"), col("order_item_product_price_fixed").cast("bigint"))
    )

In [0]:
recovered_df = recovered_df.drop("order_item_id_fixed", "order_item_order_id_fixed", "order_item_product_id_fixed", "order_item_quantity_fixed", "order_item_subtotal_fixed", "order_item_product_price_fixed")

In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-8 : APPLY DATA QUALITY RULES
# =========================================================

cleaned_recovered_df = recovered_df.filter(
        col("order_item_order_id").isNotNull() &
        col("order_item_product_id").isNotNull() &
        col("order_item_quantity").isNotNull() &
        col("order_item_subtotal").isNotNull() &
        col("order_item_product_price").isNotNull() &
        (col("order_item_quantity") >= 0) &
        (col("order_item_subtotal") >= 0) &
        (col("order_item_product_price") >= 0)
)

In [0]:
cleaned_recovered_df = cleaned_recovered_df.dropDuplicates(["order_item_order_id"])
cleaned_recovered_df.createOrReplaceTempView("order_items_cleaned_vw_fixed")

In [0]:
from pyspark.sql.functions import when, current_timestamp, sha2, concat_ws

cleaned_recovered_df = cleaned_recovered_df \
    .withColumn("order_item_quantity", col("order_item_quantity").cast("integer")) \
    .withColumn("order_item_subtotal", col("order_item_subtotal").cast("decimal(10,2)")) \
    .withColumn("order_item_product_price", col("order_item_product_price").cast("bigint")) \
    .withColumn("batch_id", col("batch_id").cast("integer")) \
    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False))

cleaned_recovered_df = cleaned_recovered_df.select("order_item_id", "order_item_order_id", "order_item_product_id", "order_item_quantity", "order_item_subtotal", "order_item_product_price", "op", "is_deleted", "source_system", "source_file_name", "ingestion_ts", "ingestion_dt", "batch_id", "run_id")

cleaned_recovered_df = cleaned_recovered_df.withColumn("event_ts", current_timestamp()) \
                    .withColumn("record_hash",
                                sha2(
                                    concat_ws(
                                        "||",
                                        col("order_item_id"),
                                        col("order_item_order_id"),
                                        col("order_item_product_id"),
                                        col("order_item_quantity"),
                                        col("order_item_subtotal"),
                                        col("order_item_product_price")
                                    ),
                                    256
                                )
                            )
    

try:
    cleaned_recovered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("retails.silver.order_items_cdc")

except Exception as e:
    print(str(e))

In [0]:
merge_query_rescued = """
    MERGE INTO retails.silver.order_items_quarantine t
    USING order_items_cleaned_vw_fixed s
    ON t.order_item_id = s.order_item_id
    WHEN MATCHED THEN
        UPDATE SET t.quarantine_status = 'FIXED', t.reprocessed_at = current_timestamp()
 
    """
spark.sql(merge_query_rescued).show()


In [0]:
update_query_corrupt = """
        UPDATE retails.silver.order_items_quarantine
        SET
            quarantine_status = 'INVALID',
            reprocessed_at = current_timestamp()
        WHERE quarantine_status = 'NEW'
"""

spark.sql(update_query_corrupt).show()

In [0]:
%sql
-- select * from retails.silver.order_items_quarantine;
-- select * from retails.silver.order_items_cdc;


In [0]:
# %sql
# ALTER TABLE retails.silver.order_items_quarantine
# ADD COLUMNS (
#     reprocessed_at TIMESTAMP
# )